# ロジスティック回帰と閾値

**出典**　機械学習 4 分野の教科書 — 第 3 章 1-B 分類 ／ 02 ロジスティック回帰：分類の出発点

**このノートのパス**　`ml-code/03_classification/01_logistic_regression.ipynb`

**このノートで確かめること**
確率を出力する分類器を学習し、**閾値を変えると陽性判定の件数が変わる**ことを確かめる。
この「閾値は後から動かせる」性質が、1-D の閾値最適化につながる。

| 項目 | 内容 |
|---|---|
| 必要なパッケージ | `scikit-learn, numpy` |
| 実行時間の目安 | 約 5 秒 |
| 前提 | **このノート単体で完結する**（他のノートを先に実行する必要はない） |

In [2]:
# 未導入なら実行する（Colab など）
# !pip install -q scikit-learn numpy

In [3]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X, y = load_breast_cancer(return_X_y=True)      # sklearn 同梱。オフラインで動く
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0)   # stratify でクラス比を保つ

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(C=1.0, max_iter=1000)   # C は正則化の強さの逆数（小さいほど強い正則化）
).fit(X_tr, y_tr)

proba = clf.predict_proba(X_te)[:, 1]          # クラス 1 である確率
print(classification_report(y_te, clf.predict(X_te)))
print("ROC-AUC:", roc_auc_score(y_te, proba))

              precision    recall  f1-score   support

           0       1.00      0.95      0.98        42
           1       0.97      1.00      0.99        72

    accuracy                           0.98       114
   macro avg       0.99      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

ROC-AUC: 0.9957010582010581


### 閾値を変えて運用を調整できるのが確率出力の利点

In [4]:
for th in [0.3, 0.5, 0.7]:
    pred = (proba >= th).astype(int)
    print(f"閾値 {th}: 陽性と判定した件数 = {pred.sum()}")

閾値 0.3: 陽性と判定した件数 = 76
閾値 0.5: 陽性と判定した件数 = 74
閾値 0.7: 陽性と判定した件数 = 71


## 読み取りどころ

**モデルは 1 つなのに、運用は閾値の数だけ作れる。**再学習せずに、
「取りこぼしを減らしたい」「誤報を減らしたい」を切り替えられる。

⚠️ ただし `predict()` の既定は 0.5 固定である。**0.5 は「決めなかった」という意味であって「最適」という意味ではない**——コストから閾値を決める手順は `01D_hpo/02_threshold_optimization.ipynb` で扱う。